# 05 - Robust (degradation-aware) training and comparison
Runtime: **T4 GPU** for the training cells. The evaluation, robustness and comparison
cells also work on a CPU runtime later. Run the cells **one at a time**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from google.colab import userdata

try:
    token = userdata.get('GH_TOKEN')  # only needed while the repo is private
except Exception:
    token = None
AUTH = f'{token}@' if token else ''
REPO = 'ThemistoklisSakkas/sar-ship-robustness'
DEST = '/content/sar-ship-robustness'
if os.path.isdir(DEST):
    !git -C {DEST} pull -q https://{AUTH}github.com/{REPO}.git main
else:
    !git clone -q https://{AUTH}github.com/{REPO}.git {DEST}
    !git -C {DEST} remote set-url origin https://github.com/{REPO}.git
%cd {DEST}
!pip install -q -r requirements.txt

DRIVE_ROOT = '/content/drive/MyDrive/sar-ship-robustness'
DATA_ROOT = f'{DRIVE_ROOT}/data'
RUNS_ROOT = f'{DRIVE_ROOT}/runs'
NAME = 'robust_yolo11n'
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Copy the dataset to the local disk (1-2 min, once per runtime)
Reading images from Google Drive is ~100x slower than the local disk and made the
first epoch take 6 min. Training reads the local copy; results still go to Drive.

In [ ]:
LOCAL = '/content/data_local'
!mkdir -p {LOCAL} && cp -r {DATA_ROOT}/ssdd_yolo {LOCAL}/
!sed -i "s#^path: .*#path: {LOCAL}/ssdd_yolo#" {LOCAL}/ssdd_yolo/*.yaml
!rm -f {LOCAL}/ssdd_yolo/labels/*.cache
!grep "^path" {LOCAL}/ssdd_yolo/ssdd.yaml
!ls {LOCAL}/ssdd_yolo/images/train | wc -l

## Step 1 - smoke test with a control (about 2 x 1 min on GPU)
Two identical 1-epoch runs, one WITH the degradation augmentation and one WITHOUT
(control). Same seed, same data: any difference comes from the augmentation alone.

In [ ]:
!python -m sarship.train --data-root {LOCAL} --runs-root {RUNS_ROOT} --name smoke_robust --overwrite --set robust_train.enabled=true train.epochs=1 2>&1 | grep -E "Degradation-aware|1/1|  all  "
!python -m sarship.train --data-root {LOCAL} --runs-root {RUNS_ROOT} --name smoke_control --overwrite --set train.epochs=1 2>&1 | grep -E "Degradation-aware|1/1|  all  "

In [ ]:
from IPython.display import Image, display
for run in ['smoke_robust', 'smoke_control']:
    print(run)
    display(Image(f'{RUNS_ROOT}/{run}/train_batch0.jpg', width=700))

## Step 2 - full robust training (30-60 min on GPU)
After a disconnect: re-run the setup cells (Drive, setup, local copy), then the resume cell.

In [ ]:
!python -m sarship.train --data-root {LOCAL} --runs-root {RUNS_ROOT} --name {NAME} --set robust_train.enabled=true

In [ ]:
# Only after a disconnect (the saved config of the run is reused automatically):
# !python -m sarship.train --data-root {LOCAL} --runs-root {RUNS_ROOT} --name {NAME} --resume

## Step 3 - evaluation, robustness study, comparison (CPU is fine)

In [ ]:
!python -m sarship.evaluate --weights {RUNS_ROOT}/{NAME}/weights/best.pt --data-root {LOCAL} --runs-root {RUNS_ROOT} --name {NAME}

In [ ]:
!python -m sarship.robustness --weights {RUNS_ROOT}/{NAME}/weights/best.pt --data-root {LOCAL} --runs-root {RUNS_ROOT} --name {NAME} --work-dir /content/degraded_robust

In [ ]:
!python -m sarship.compare --data-root {LOCAL} --runs-root {RUNS_ROOT}

In [ ]:
for path in [f'{RUNS_ROOT}/{NAME}/evaluation/training_curves.png',
             f'{RUNS_ROOT}/{NAME}/comparison/comparison.png']:
    print(path.split('/')[-1])
    display(Image(path))